# Stock and newsreader agent
Last update: _2026.02.28_<br/>

## Configuration

In [2]:
import os
from pathlib import Path

In [3]:
def load_to_env_var(filename: str, env_var: str) -> None:
    token_path = Path(filename)
    if not token_path.exists():
        raise FileNotFoundError(
            "Missing {:}. Create it with your Hugging Face token on a single line.".format(filename)
        )

    token = token_path.read_text(encoding="utf-8").strip()
    if not token:
        raise ValueError("{:} is empty.".format(filename))
    
    os.environ[env_var] = token

In [ ]:
load_to_env_var('.openai_api_key.txt', 'OPENAI_API_KEY')

FileNotFoundError: Missing configs/.openai_api_key.txt. Create it with your Hugging Face token on a single line.

In [4]:
os.environ["OPENAI_LOG"] = "debug"

## Tools

In [5]:
from dataclasses import dataclass

from langchain.agents import create_agent

from langchain_openai import ChatOpenAI
from langchain.tools import tool, ToolRuntime

In [6]:
SYSTEM_PROMPT = """You are an expert in stock and news analysis. You have access to the following tools:

- get_stock_value: use this to get the stock value for a specific company or stock ticker
- fetch_stock_news: use this to fetch news for a specific company

Call get_stock_value and fetch_stock_news at most once per question. After you receive the responses, respond with a recommendation and STOP. If you don't have enough information to provide a recommendation, provide a generic recommendation based on the stock value and the recent news and STOP."""

### Stock reader

In [7]:
from __future__ import annotations

import re
import html
import math

import requests
import datetime as dt

from urllib.parse import quote_plus
import xml.etree.ElementTree as ET

from typing import Any, Dict, List, Optional

In [8]:
_YAHOO_CHART_URL = "https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
_DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; AgentTool/1.0; +https://example.com/bot)",
    "Accept": "application/json,text/plain,*/*",
}

In [9]:
@dataclass
class Stock:
    symbol: str
    currency: Optional[str]
    as_of: str
    value: float
    changes_pct: Dict[str, Optional[float]]
    source: str

In [10]:
"""
yahoo_price_tool.py

Agent-tool function:
  get_value_and_pct_changes(symbol: str) -> dict

Returns:
  {
    "symbol": "AAPL",
    "currency": "USD",
    "as_of": "2026-02-28T12:34:56Z",
    "value": 182.34,
    "changes_pct": {
        "1m":  3.21,
        "3m": -1.05,
        "6m":  9.87
    },
    "source": "yahoo_finance_chart_v8"
  }

Notes:
- Month lookbacks use trading-day approximations:
    1m ~ 21 trading days, 3m ~ 63, 6m ~ 126, 12m ~ 252
- Handles missing data by returning None for that window.
"""

def _utc_now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def _pct_change(current: float, past: float) -> Optional[float]:
    if past is None or past == 0 or current is None:
        return None
    return (current / past - 1.0) * 100.0


def _nearest_prior_close(timestamps: list[int], closes: list[Optional[float]], target_ts: int) -> Optional[float]:
    """
    Find the close price at or before target_ts.
    We walk from the end for speed (timestamps are ascending).
    """
    for i in range(len(timestamps) - 1, -1, -1):
        if timestamps[i] <= target_ts:
            c = closes[i]
            if c is not None and not (isinstance(c, float) and math.isnan(c)):
                return float(c)
    return None


def _fetch_yahoo_chart(symbol: str, session: Optional[requests.Session] = None) -> Dict[str, Any]:
    """
    Fetch up to 1y daily data so we can compute 1/3/6 month changes.
    """
    s = session or requests.Session()
    url = _YAHOO_CHART_URL.format(symbol=symbol)
    params = {
        "range": "1y",
        "interval": "1d",
        "includePrePost": "false",
        "events": "div,splits",
    }
    resp = s.get(url, params=params, headers=_DEFAULT_HEADERS, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    # Basic validation
    chart = data.get("chart", {})
    if chart.get("error"):
        raise ValueError(f"Yahoo chart error for {symbol}: {chart['error']}")
    result = (chart.get("result") or [None])[0]
    if not result:
        raise ValueError(f"No chart result for {symbol}")

    return result

@tool
def get_stock_value(symbol: str) -> str:
    """
    Reads the current stock price for the given symbol and computes percentage changes over 1m/3m/6m lookbacks.
    """
    symbol = symbol.strip()
    if not symbol:
        raise ValueError("symbol must be non-empty, e.g. 'AAPL' or 'MSFT' or 'BTC-USD'")

    result = _fetch_yahoo_chart(symbol)
    meta = result.get("meta", {}) or {}

    timestamps = result.get("timestamp") or []
    indicators = (result.get("indicators") or {}).get("quote") or []
    quote0 = indicators[0] if indicators else {}
    closes = quote0.get("close") or []

    if not timestamps or not closes:
        raise ValueError(f"Missing price series for {symbol}")

    # Current value: prefer meta regularMarketPrice; fallback to last non-null close
    current_value = meta.get("regularMarketPrice")
    if current_value is None:
        # last valid close
        current_value = next((float(c) for c in reversed(closes) if c is not None and not (isinstance(c, float) and math.isnan(c))), None)
    if current_value is None:
        raise ValueError(f"Could not determine current value for {symbol}")

    # Approximate trading-day lookbacks
    lookbacks = {
        "1m": 21,
        "3m": 63,
        "6m": 126
    }

    # Use timestamps to find a price near the lookback date (calendar-based), not index-based,
    # because holidays / missing days can shift counts.
    # We compute target_ts by subtracting N calendar days that roughly correspond to the trading-day window.
    # (This is intentionally simple and robust.)
    calendar_day_map = {
        "1m": 31,
        "3m": 93,
        "6m": 186,
    }

    latest_ts = int(timestamps[-1])
    changes: Dict[str, Optional[float]] = {}

    for k in ("1m", "3m", "6m"):
        target_ts = latest_ts - calendar_day_map[k] * 24 * 3600
        past_close = _nearest_prior_close(timestamps, closes, target_ts)
        changes[k] = None if past_close is None else round(_pct_change(float(current_value), float(past_close)), 4)

    return Stock(
        symbol=symbol,
        currency=meta.get("currency"),
        as_of=_utc_now_iso(),
        value=float(current_value),
        changes_pct=changes,
        source="yahoo_finance_chart_v8",
    )

In [11]:
get_stock_value.invoke({"symbol": "AAPL"})

Stock(symbol='AAPL', currency='USD', as_of='2026-03-01T08:14:13Z', value=264.18, changes_pct={'1m': 2.2883, '3m': -4.8171, '6m': 16.2969}, source='yahoo_finance_chart_v8')

In [12]:
get_stock_value.invoke({"symbol": "YEXT"})

Stock(symbol='YEXT', currency='USD', as_of='2026-03-01T08:14:13Z', value=5.68, changes_pct={'1m': -24.2667, '3m': -32.6216, '6m': -37.2376}, source='yahoo_finance_chart_v8')

### Google news

In [13]:
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; AgentTool/1.0; +https://example.com/bot)",
    "Accept": "application/rss+xml,application/xml;q=0.9,*/*;q=0.8",
}

In [14]:
@dataclass
class NewsItem:
    title: str
    url: str
    publisher: str
    published: Optional[str]
    summary: str
    source: str

    def to_dict(self) -> Dict[str, Any]:
        return {
            "title": self.title,
            "url": self.url,
            "publisher": self.publisher,
            "published": self.published,
            "summary": self.summary,
            "source": self.source,
        }

In [15]:
"""
stock_news_tool.py

StockNewsReader: fetches recent news for a stock ticker and/or company query.

Sources:
- Yahoo Finance RSS (ticker):  https://feeds.finance.yahoo.com/rss/2.0/headline?s=AAPL&region=US&lang=en-US
- Google News RSS (query):     https://news.google.com/rss/search?q=Apple%20stock&hl=en-US&gl=US&ceid=US:en

No API keys required.

Output format:
{
  "query": "AAPL",
  "as_of": "2026-02-28T12:34:56Z",
  "items": [
    {
      "title": "...",
      "url": "...",
      "publisher": "...",
      "published": "2026-02-27T09:10:00Z",
      "summary": "...",
      "source": "yahoo_rss" | "google_news_rss"
    },
    ...
  ]
}
"""

def _parse_rfc2822_to_iso(s: str) -> Optional[str]:
    """
    RSS pubDate is often RFC 2822 (e.g., 'Fri, 16 Feb 2026 14:22:00 GMT').
    """
    if not s:
        return None
    s = s.strip()
    # dt.email.utils parsedate_to_datetime is clean, but keep deps minimal:
    try:
        from email.utils import parsedate_to_datetime

        d = parsedate_to_datetime(s)
        if d.tzinfo is None:
            d = d.replace(tzinfo=dt.timezone.utc)
        return d.astimezone(dt.timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
    except Exception:
        return None


def _strip_html(text: str) -> str:
    if not text:
        return ""
    text = html.unescape(text)
    # remove tags
    text = re.sub(r"<[^>]+>", " ", text)
    # normalize whitespace
    return re.sub(r"\s+", " ", text).strip()


def _safe_text(elem: Optional[ET.Element]) -> str:
    if elem is None or elem.text is None:
        return ""
    return elem.text.strip()


class StockNewsReader:
    def __init__(
        self,
        session: Optional[requests.Session] = None,
        timeout_s: int = 20,
        headers: Optional[Dict[str, str]] = None,
    ) -> None:
        self.session = session or requests.Session()
        self.timeout_s = timeout_s
        self.headers = headers or DEFAULT_HEADERS.copy()

    # ---------- Public API ----------

    def fetch_news(
        self,
        ticker: Optional[str] = None,
        company: Optional[str] = None,
        limit: int = 20,
        include_google: bool = True,
        include_yahoo: bool = True,
    ) -> Dict[str, Any]:
        """
        Fetch news items from enabled sources, dedupe by URL, and return newest-first where possible.
        """
        if not ticker and not company:
            raise ValueError("Provide at least one of: ticker='AAPL' or company='Apple'")

        items: List[NewsItem] = []

        if include_yahoo and ticker:
            items.extend(self._fetch_yahoo_rss(ticker=ticker, limit=limit))

        if include_google:
            # If company not provided, use ticker as query; if both exist, combine.
            query = self._build_google_query(ticker=ticker, company=company)
            items.extend(self._fetch_google_news_rss(query=query, limit=limit))

        # Dedupe by URL (keep first occurrence)
        seen = set()
        deduped: List[NewsItem] = []
        for it in items:
            if not it.url or it.url in seen:
                continue
            seen.add(it.url)
            deduped.append(it)

        # Sort: newest first when published is parseable; otherwise stable
        def sort_key(it: NewsItem):
            if not it.published:
                return (0, "")  # older bucket
            return (1, it.published)

        deduped.sort(key=sort_key, reverse=True)
        deduped = deduped[:limit]

        return {
            "query": company or ticker or "",
            "as_of": _utc_now_iso(),
            "items": [it.to_dict() for it in deduped],
        }

    # ---------- Yahoo Finance RSS ----------

    def _fetch_yahoo_rss(self, ticker: str, limit: int) -> List[NewsItem]:
        # Example:
        # https://feeds.finance.yahoo.com/rss/2.0/headline?s=AAPL&region=US&lang=en-US
        url = (
            "https://feeds.finance.yahoo.com/rss/2.0/headline"
            f"?s={quote_plus(ticker.strip())}&region=US&lang=en-US"
        )
        xml = self._get_text(url)
        return self._parse_rss_items(xml, source="yahoo_rss", default_publisher="Yahoo Finance", limit=limit)

    # ---------- Google News RSS ----------

    def _build_google_query(self, ticker: Optional[str], company: Optional[str]) -> str:
        parts = []
        if company:
            parts.append(company.strip())
        if ticker and (not company or ticker.strip().lower() not in company.strip().lower()):
            parts.append(ticker.strip())
        # Make it more finance-relevant without overconstraining:
        parts.append("stock OR shares OR earnings OR revenue OR guidance")
        return " ".join(parts)

    def _fetch_google_news_rss(self, query: str, limit: int) -> List[NewsItem]:
        # Example:
        # https://news.google.com/rss/search?q=Apple%20stock&hl=en-US&gl=US&ceid=US:en
        q = quote_plus(query.strip())
        url = f"https://news.google.com/rss/search?q={q}&hl=en-US&gl=US&ceid=US:en"
        xml = self._get_text(url)
        # Google often includes source in <source> under item; we’ll parse it.
        return self._parse_rss_items(xml, source="google_news_rss", default_publisher="Google News", limit=limit)

    # ---------- HTTP + RSS parsing ----------

    def _get_text(self, url: str) -> str:
        resp = self.session.get(url, headers=self.headers, timeout=self.timeout_s)
        resp.raise_for_status()
        return resp.text

    def _parse_rss_items(
        self,
        xml_text: str,
        source: str,
        default_publisher: str,
        limit: int,
    ) -> List[NewsItem]:
        """
        Parses RSS 2.0 feeds. Works for both Yahoo Finance RSS and Google News RSS.
        """
        root = ET.fromstring(xml_text)
        channel = root.find("channel")
        if channel is None:
            return []

        out: List[NewsItem] = []
        for item in channel.findall("item"):
            title = _safe_text(item.find("title"))
            link = _safe_text(item.find("link"))

            pub = _parse_rfc2822_to_iso(_safe_text(item.find("pubDate")))

            # Prefer <source> for Google News RSS if present
            src_elem = item.find("source")
            publisher = _safe_text(src_elem) if src_elem is not None else default_publisher
            publisher = publisher or default_publisher

            desc = _safe_text(item.find("description"))
            summary = _strip_html(desc)

            if not title and not link:
                continue

            out.append(
                NewsItem(
                    title=title or "",
                    url=link or "",
                    publisher=publisher,
                    published=pub,
                    summary=summary,
                    source=source,
                )
            )

            if len(out) >= limit:
                break

        return out

@tool
def fetch_stock_news(
    ticker: Optional[str] = None,
    company: Optional[str] = None,
    limit: int = 20,
    include_google: bool = True,
    include_yahoo: bool = True,
) -> Dict[str, Any]:
    """
    A thin wrapper to expose StockNewsReader as a single tool function.
    """
    reader = StockNewsReader()

    return reader.fetch_news(
        ticker=ticker,
        company=company,
        limit=limit,
        include_google=include_google,
        include_yahoo=include_yahoo,
    )

In [16]:
fetch_stock_news.invoke({"ticker": "YEXT", "company": "Yext", "limit": 5})

{'query': 'Yext',
 'as_of': '2026-03-01T08:14:16Z',
 'items': [{'title': 'Yext (NYSE:YEXT) Short Interest Update - MarketBeat',
   'url': 'https://news.google.com/rss/articles/CBMikwFBVV95cUxPSF9GeHNXcS1XZ2FTaFA3bzRKMDJjb2VtcE1TdE5EeUxWQzJDbS1ickk1Rl9jUHNRbDRuaTZwaTVfVWppM0VXdlN6ZnpyVjVNTW5wX2tUallpUGJXanFPOEFVS2hORGtWakZ4dWdZWnp3a1FpaEpXbVI0Y1MyYnIzaE80NmIzUlJhQnlEMmNLNkRLVXM?oc=5',
   'publisher': 'MarketBeat',
   'published': '2026-02-28T10:26:31Z',
   'summary': 'Yext (NYSE:YEXT) Short Interest Update MarketBeat',
   'source': 'google_news_rss'},
  {'title': 'INVESTOR ALERT: Pomerantz Law Firm Investigates Claims On Behalf of Investors of Yext, Inc. - YEXT - PR Newswire',
   'url': 'https://news.google.com/rss/articles/CBMi5gFBVV95cUxOdXRmUVhFbHZuM2dXVVNjbk5vWWVQbEd3U3BfOEtlNWZ0aTk5V2ZERFBzLW5QS2UzMVZEU0U4TXVGNGQ2ZWg2ZTRJYndTTnZPcmJiVy1UUGRXOTNNeklTc09xcE9mMHZrN1ktWm95U0JVcXJZbEc3ZmdBaXJ5RGlCWXVHN0tORUJZczJScjYtdzhvcDhBZkY3TXJtaGhoNF90VFpBME4wSmU2aEdYS3VTcUc1VzhzVVpIcDNkOVU5MXQ2MEJ

## Agent

In [17]:
## OpenAI
model = ChatOpenAI(
    model="gpt-4o-mini",  # or gpt-4o
    temperature=0
)

In [19]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [20]:
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_stock_value, fetch_stock_news],
    #response_format=ToolStrategy(ResponseFormat),
    #checkpointer=checkpointer,
    debug=True
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}, "recursion_limit": 10}

In [21]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Would you recommend investing in Yext (YEXT)?"}]},
    config=config
)

[2026-03-01 09:14:24 - openai._base_client:486 - DEBUG] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-b44d3179-d48a-4e86-a28f-3026c11543c3', 'content': None, 'json_data': {'messages': [{'content': "You are an expert in stock and news analysis. You have access to the following tools:\n\n- get_stock_value: use this to get the stock value for a specific company or stock ticker\n- fetch_stock_news: use this to fetch news for a specific company\n\nCall get_stock_value and fetch_stock_news at most once per question. After you receive the responses, respond with a recommendation and STOP. If you don't have enough information to provide a recommendation, provide a generic recommendation based on the stock value and the recent news and STOP.", 'role': 'system'}, {'content': 'Would you recommend investing in Yext (YEXT)?', 'role': 'user'}], 'model': 'gpt-4o-mini', 'stream'

[values] {'messages': [HumanMessage(content='Would you recommend investing in Yext (YEXT)?', additional_kwargs={}, response_metadata={}, id='10a38719-fada-466b-9237-d9367eee42cf')]}


[2026-03-01 09:14:26 - httpx:1025 - INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-01 09:14:26 - openai._base_client:1039 - DEBUG] HTTP Response: POST https://api.openai.com/v1/chat/completions "200 OK" Headers({'date': 'Sun, 01 Mar 2026 08:14:26 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'access-control-expose-headers': 'X-Request-ID', 'openai-organization': 'user-lwikzmeav2mgnaotzr6fjksz', 'openai-processing-ms': '1175', 'openai-project': 'proj_gfaaKRXSl5BnTxq9D6u4t3L4', 'openai-version': '2020-10-01', 'server': 'cloudflare', 'x-ratelimit-limit-requests': '10000', 'x-ratelimit-limit-tokens': '200000', 'x-ratelimit-remaining-requests': '9999', 'x-ratelimit-remaining-tokens': '199851', 'x-ratelimit-reset-requests': '8.64s', 'x-ratelimit-reset-tokens': '44ms', 'x-request-id': 'req_bcadc849314040ce8ba797410166ae18', 'x-openai-proxy-wasm': 'v0.1', 'cf-cache-status': 'DYNAMIC', 'set-c

[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 261, 'total_tokens': 309, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b2175cdf48', 'id': 'chatcmpl-DEWETuSOEtOoarIbQWH1T8eWuOQ8z', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ca876-81cd-7f41-9a4f-02e7a2aea11f-0', tool_calls=[{'name': 'get_stock_value', 'args': {'symbol': 'YEXT'}, 'id': 'call_mqCaWSuY7UbLHemuZ3gtacak', 'type': 'tool_call'}, {'name': 'fetch_stock_news', 'args': {'ticker': 'YEXT'}, 'id': 'call_4lk4jVMbH4WEN8DjwJtZrypZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 261, 'outp

[2026-03-01 09:14:27 - openai._base_client:486 - DEBUG] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-b8bc29cd-b3aa-46e8-8d2d-7ff2ddad6d45', 'content': None, 'json_data': {'messages': [{'content': "You are an expert in stock and news analysis. You have access to the following tools:\n\n- get_stock_value: use this to get the stock value for a specific company or stock ticker\n- fetch_stock_news: use this to fetch news for a specific company\n\nCall get_stock_value and fetch_stock_news at most once per question. After you receive the responses, respond with a recommendation and STOP. If you don't have enough information to provide a recommendation, provide a generic recommendation based on the stock value and the recent news and STOP.", 'role': 'system'}, {'content': 'Would you recommend investing in Yext (YEXT)?', 'role': 'user'}, {'content': None, 'role': 'assist

[updates] {'tools': {'messages': [ToolMessage(content='{"query": "YEXT", "as_of": "2026-03-01T08:14:27Z", "items": [{"title": "Yext (NYSE:YEXT) Short Interest Update - MarketBeat", "url": "https://news.google.com/rss/articles/CBMikwFBVV95cUxPSF9GeHNXcS1XZ2FTaFA3bzRKMDJjb2VtcE1TdE5EeUxWQzJDbS1ickk1Rl9jUHNRbDRuaTZwaTVfVWppM0VXdlN6ZnpyVjVNTW5wX2tUallpUGJXanFPOEFVS2hORGtWakZ4dWdZWnp3a1FpaEpXbVI0Y1MyYnIzaE80NmIzUlJhQnlEMmNLNkRLVXM?oc=5", "publisher": "MarketBeat", "published": "2026-02-28T10:26:31Z", "summary": "Yext (NYSE:YEXT) Short Interest Update MarketBeat", "source": "google_news_rss"}, {"title": "INVESTOR ALERT: Pomerantz Law Firm Investigates Claims On Behalf of Investors of Yext, Inc. - YEXT - PR Newswire", "url": "https://news.google.com/rss/articles/CBMi5gFBVV95cUxOdXRmUVhFbHZuM2dXVVNjbk5vWWVQbEd3U3BfOEtlNWZ0aTk5V2ZERFBzLW5QS2UzMVZEU0U4TXVGNGQ2ZWg2ZTRJYndTTnZPcmJiVy1UUGRXOTNNeklTc09xcE9mMHZrN1ktWm95U0JVcXJZbEc3ZmdBaXJ5RGlCWXVHN0tORUJZczJScjYtdzhvcDhBZkY3TXJtaGhoNF90VFpBME4wSmU2aE

[2026-03-01 09:14:32 - httpx:1025 - INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2026-03-01 09:14:32 - openai._base_client:1039 - DEBUG] HTTP Response: POST https://api.openai.com/v1/chat/completions "200 OK" Headers({'date': 'Sun, 01 Mar 2026 08:14:32 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'access-control-expose-headers': 'X-Request-ID', 'openai-organization': 'user-lwikzmeav2mgnaotzr6fjksz', 'openai-processing-ms': '4607', 'openai-project': 'proj_gfaaKRXSl5BnTxq9D6u4t3L4', 'openai-version': '2020-10-01', 'server': 'cloudflare', 'x-ratelimit-limit-requests': '10000', 'x-ratelimit-limit-tokens': '200000', 'x-ratelimit-remaining-requests': '9998', 'x-ratelimit-remaining-tokens': '196756', 'x-ratelimit-reset-requests': '14.694s', 'x-ratelimit-reset-tokens': '973ms', 'x-request-id': 'req_2530b460d52643f1a2a2a6908f65a2bc', 'x-openai-proxy-wasm': 'v0.1', 'cf-cache-status': 'DYNAMIC', 'x-

[updates] {'model': {'messages': [AIMessage(content="Yext (YEXT) is currently trading at $5.68, which reflects a significant decline of approximately 24.27% over the past month, 32.62% over the last three months, and 37.24% over the past six months. This downward trend indicates a challenging period for the stock.\n\nRecent news highlights several key points:\n1. There is ongoing scrutiny regarding Yext, with a law firm investigating claims on behalf of investors, which could indicate potential legal issues or concerns about the company's governance.\n2. Yext has announced a tender offer to purchase up to $180 million of its common stock, which could be seen as a positive move to enhance shareholder value.\n3. The company is also focusing on strategic partnerships, such as with AdCellerant, to improve brand visibility, which may help in its recovery.\n\nGiven the significant decline in stock value and the potential legal investigations, I would recommend exercising caution before inves

In [22]:
print(response['messages'][-1].content)

Yext (YEXT) is currently trading at $5.68, which reflects a significant decline of approximately 24.27% over the past month, 32.62% over the last three months, and 37.24% over the past six months. This downward trend indicates a challenging period for the stock.

Recent news highlights several key points:
1. There is ongoing scrutiny regarding Yext, with a law firm investigating claims on behalf of investors, which could indicate potential legal issues or concerns about the company's governance.
2. Yext has announced a tender offer to purchase up to $180 million of its common stock, which could be seen as a positive move to enhance shareholder value.
3. The company is also focusing on strategic partnerships, such as with AdCellerant, to improve brand visibility, which may help in its recovery.

Given the significant decline in stock value and the potential legal investigations, I would recommend exercising caution before investing in Yext. The tender offer and strategic partnerships co

In [23]:
len(response['messages'])

5

__Todo__
- streamlit/FastAPI app __done__
- Unit test __done__
- monitoring/evaluation
- Deployment plan